In [ ]:
import sys
import gc
import json
import logging
from pathlib import Path

import torch

# Repo root import (works in local + Colab)
sys.path.append("./Efficient_Architecture")

from src.utils.models import Qwen3, Qwen35, LFM2, IBM_Granite1b, IBM_Granite
from data.preprocessing import c4_dataset
from src.test import test_model
from test_vectors import test_suite
from src.utils.metrics import non_embedding_params, count_params

# Choose dataset configuration
LANGUAGE = "en"
SPLIT = "train"

OUT_DIR = Path("metrics")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = OUT_DIR / "output.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_PATH), logging.StreamHandler()],
)
logger = logging.getLogger("benchmark")

logger.info("Starting main.ipynb benchmark data generation")

# Baseline peak memory for incremental memory plots
BASELINE_READ_LEN = 64  # N0
# G0 is the smallest gen_len used at N0 (usually 16)
baseline_gen_lens = sorted({g for (r, g) in test_suite if r == BASELINE_READ_LEN})
BASELINE_GEN_LEN = baseline_gen_lens[0] if baseline_gen_lens else 16
BASELINE_KEY = f"({BASELINE_READ_LEN},{BASELINE_GEN_LEN})"

logger.info(f"Using baseline memory key: {BASELINE_KEY}")

BASELINE_JSON_PATH = OUT_DIR / "baseline_peak_mem_gb_by_model.json"
baseline_peak_mem_gb_by_model = {}

# Model name -> class (instantiate one at a time to limit memory)
model_classes = {
    "Qwen3": Qwen3,
    "Qwen3.5": Qwen35,
    "LFM2": LFM2,
    "IBM-G1B": IBM_Granite1b,
    "IBM-G350M": IBM_Granite,
}

results_files = []

for name, ModelClass in model_classes.items():
    logger.info(f"\n=== {name} ===")

    wrapper = None
    dataset = None
    metrics = None
    json_path = OUT_DIR / f"{name}_metrics.json"
    results_files.append(str(json_path))

    try:
        wrapper = ModelClass()
        total_p = count_params(wrapper.model)
        non_emb_p = non_embedding_params(wrapper.model)
        logger.info(
            f"Parameter count: total={total_p:,} | non-embedding={non_emb_p:,} | embedding+head={total_p - non_emb_p:,}"
        )

        logger.info(f"Running metrics for {name}...")
        dataset = c4_dataset(split=SPLIT, language=LANGUAGE, tokenizer=wrapper.tokenizer)
        metrics = test_model(wrapper.model, dataset, json_path=str(json_path))

        # Save raw metrics JSON (final dump after completion; incremental dumps already happened)
        with open(json_path, "w") as f:
            serializable = {}
            for (read_len, gen_len), vals in metrics.items():
                serializable[f"({read_len},{gen_len})"] = vals
            json.dump(serializable, f)

        logger.info(f"Saved metrics to {json_path}")

        # Extract baseline peak memory for incremental memory plots
        base_mem = None
        for (read_len, gen_len), vals in metrics.items():
            if read_len == BASELINE_READ_LEN and gen_len == BASELINE_GEN_LEN:
                if isinstance(vals, dict) and "Peak Mem." in vals:
                    base_mem = vals["Peak Mem."]
                break

        baseline_peak_mem_gb_by_model[name] = base_mem
        logger.info(f"Baseline peak memory {BASELINE_KEY} for {name}: {base_mem}")

    except torch.cuda.OutOfMemoryError as e:
        logger.error(f"CUDA OOM for {name}. test_model should have saved partial results; leaving file as-is. Error: {e}")
        baseline_peak_mem_gb_by_model[name] = None
        if not Path(json_path).exists():
            with open(json_path, "w") as f:
                json.dump({}, f)

    except RuntimeError as e:
        msg = str(e).lower()
        if "out of memory" in msg or ("cuda" in msg and "memory" in msg):
            logger.error(f"RuntimeError OOM for {name}. test_model should have saved partial results; leaving file as-is. Error: {e}")
            baseline_peak_mem_gb_by_model[name] = None
            if not Path(json_path).exists():
                with open(json_path, "w") as f:
                    json.dump({}, f)
        else:
            logger.error(f"RuntimeError for {name}: {e}. test_model may not have saved partial results.")
            baseline_peak_mem_gb_by_model[name] = None
            if not Path(json_path).exists():
                with open(json_path, "w") as f:
                    json.dump({}, f)

    except Exception as e:
        logger.exception(f"Unexpected error for {name}. test_model may not have saved partial results: {e}")
        baseline_peak_mem_gb_by_model[name] = None
        if not Path(json_path).exists():
            with open(json_path, "w") as f:
                json.dump({}, f)

    finally:
        if metrics is not None:
            del metrics
        if dataset is not None:
            del dataset
        if wrapper is not None:
            del wrapper

        gc.collect()
        if torch.cuda.is_available():
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass

# Write baseline JSON once all models have been attempted
with open(BASELINE_JSON_PATH, "w") as f:
    json.dump(
        {
            "baseline_read_len": BASELINE_READ_LEN,
            "baseline_gen_len": BASELINE_GEN_LEN,
            "baseline_key": BASELINE_KEY,
            "baseline_peak_mem_gb_by_model": baseline_peak_mem_gb_by_model,
        },
        f,
        indent=2,
    )

logger.info(f"Wrote baseline peak memory JSON to {BASELINE_JSON_PATH}")
logger.info("Finished main.ipynb benchmark data generation")


In [ ]:
import sys
from pathlib import Path
import json
import logging

import torch

# Robust repo-root import (works in local + Colab)
sys.path.append("./Efficient_Architecture")

import importlib
import test_vectors as tv
importlib.reload(tv)

test_suite = tv.test_suite
logger.info(f"Loaded test_suite for plotting: len={len(test_suite)} | suite={test_suite}")

from src.utils.visuals import generate_incremental_memory_plots, generate_plots

OUT_DIR = Path("metrics")
BASELINE_JSON_PATH = OUT_DIR / "baseline_peak_mem_gb_by_model.json"
LOG_PATH = OUT_DIR / "output.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_PATH), logging.StreamHandler()],
)
logger = logging.getLogger("benchmark_plot")

logger.info("Starting plotting cell (incremental memory + baseline table)")

# Load baseline JSON
with open(BASELINE_JSON_PATH, "r") as f:
    baseline_obj = json.load(f)

baseline_peak_mem_gb_by_model = baseline_obj.get("baseline_peak_mem_gb_by_model", {})
baseline_read_len = baseline_obj.get("baseline_read_len", 64)
baseline_gen_len = baseline_obj.get("baseline_gen_len", 16)
base_key_str = baseline_obj.get("baseline_key", f"({baseline_read_len},{baseline_gen_len})")

# Load all metrics JSONs
json_files = sorted(str(p) for p in OUT_DIR.glob("*_metrics.json"))
if not json_files:
    raise FileNotFoundError(f"No metrics json files found in {OUT_DIR}")

logger.info(f"Found {len(json_files)} metrics JSON files")

# Generate incremental-memory + other-metrics plot
plot_stem = str(OUT_DIR / "plots_incremental")
generate_incremental_memory_plots(
    json_files=json_files,
    plot_name=plot_stem,
    baseline_json_path=str(BASELINE_JSON_PATH),
    suite=test_suite,
)
logger.info(f"Saved incremental plots to {plot_stem}.png")

# Generate full peak memory plots
plot_stem_peak = str(OUT_DIR / "plots_peak")
generate_plots(
    json_files=json_files,
    plot_name=plot_stem_peak,
    suite=test_suite,
)
logger.info(f"Saved peak-memory plots to {plot_stem_peak}.png")

# Build and print a baseline table
rows = []
for model_name, mem_val in baseline_peak_mem_gb_by_model.items():
    rows.append({"Model": model_name, f"M{base_key_str} (GB)": mem_val})

try:
    import pandas as pd

    df = pd.DataFrame(rows).set_index("Model").sort_index()
    print("\nBaseline peak memory table")
    print(df)
    logger.info("Baseline peak memory table:\n%s", df.to_string())
except Exception:
    # Fallback: simple text table
    print("\nBaseline peak memory table")
    for r in rows:
        print(f"- {r['Model']}: {r[f'M{base_key_str} (GB)']}")
    logger.info("Baseline peak memory table: %s", rows)

logger.info("Finished plotting cell")
with open(LOG_PATH, "a") as f:
    f.write("Finished plotting cell\n")
